# Chinee apple: supervised probe on frozen DINOv3

Stage 2 of the *model first, labels later* plan. A logistic-regression head is
fitted on **frozen** DINOv3 patch tokens using the CVAT + SAM3 polygon
annotations, then scored at three levels: patch, mask and instance (COCO mAP).

The backbone is never fine-tuned. Only the head is trained, so this runs on a
free Colab T4 in a few minutes.

**What you need**
1. The annotated tiles, and the CVAT `batch_*.zip` exports in the same folder.
2. Optionally DINOv3 weights. They are gated, so the notebook falls back to
   ungated DINOv2 and everything downstream is identical.

## 1. Environment

In [ ]:
!nvidia-smi -L || echo "no GPU, this still runs on CPU but slower"
!pip -q install scikit-learn joblib pillow scipy
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 2. Code

Clone the repo so `train_probe.py` and the shared `backbones` and `metrics` packages are
importable. Replace the URL if you work from a fork.

In [ ]:
import os, sys
REPO = "https://github.com/nutboltu/chinee-apple-cv-benchmark.git"
if not os.path.exists("chinee-apple-cv-benchmark"):
    !git clone -q $REPO
sys.path.insert(0, "chinee-apple-cv-benchmark/pipelines/dinov3-linear-probe")
sys.path.insert(0, "chinee-apple-cv-benchmark")   # for the shared metrics package
from backbones import dino as vf
import train_probe as tp
from metrics import segmentation as pe
print("ok")

## 3. Data

The imagery is not in git. Point `DATA` at a folder holding the tile jpgs
(in `batch_01/`.. subfolders or flat) **and** the `batch_*.zip` CVAT exports.
Easiest is Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA = Path('/content/drive/MyDrive/chinee-apple/processed')   # <- edit me
CRIB = DATA / 'tile_markers.csv'        # optional, masks cardboard out

assert DATA.exists(), f"not found: {DATA}"
print("zips :", len(list(DATA.glob('batch_*.zip'))))
print("tiles:", len(list(DATA.rglob('*.jpg'))))

## 4. Backbone

Defaults to **DINOv3 ViT-B/16**, loaded from your own gated weights on Drive.

Loading DINOv3 needs **two** things, not one. The weights `.pth`, and a local
clone of Meta's `dinov3` repo, because the model is built through
`torch.hub.load(..., source='local')`. The repo supplies the architecture, the
`.pth` supplies the learned parameters. Having only the weights is not enough.

Set `FAMILY='dinov2'` to fall back to the ungated ViT-B/14 from torch.hub. The
token shape is the same, so everything downstream works either way.

In [ ]:
FAMILY   = 'dinov3'
IMG_SIZE = 448        # 448 is a good speed/detail balance, 768 is sharper and ~3x slower

# Your gated weights on Drive. Adjust if you put them elsewhere.
WEIGHTS  = '/content/drive/MyDrive/dinov3/weights/dinov3_vitb16_pretrain_lvd1689m.pth'
REPO_DIR = '/content/dinov3'      # Meta's code, cloned below. Not the weights.

# Meta's dinov3 hubconf imports its segmentation and dinotxt heads, which drag
# in dependencies Colab does not ship. Verified by reproducing the import:
# torchmetrics first, then termcolor. omegaconf and ftfy are in their
# requirements.txt and installed here pre-emptively.
!pip -q install torchmetrics termcolor omegaconf ftfy
import os
if FAMILY == 'dinov3':
    if not os.path.isdir(REPO_DIR):
        !git clone -q https://github.com/facebookresearch/dinov3.git {REPO_DIR}
    assert os.path.isfile(WEIGHTS), (
        f"weights not found at {WEIGHTS}\n"
        "Check the Drive path, and that Drive is mounted (section 3).")
    print(f"weights {os.path.getsize(WEIGHTS)/1e6:.0f} MB")
else:
    REPO_DIR = WEIGHTS = None     # DINOv2 pulls itself from torch.hub

model, meta = vf.load_backbone(family=FAMILY, repo=REPO_DIR, weights=WEIGHTS)
print(meta)

# patch size differs between families, which changes the token grid:
#   DINOv3 ViT-B/16 at 448px -> 28 x 28 tokens
#   DINOv2 ViT-B/14 at 448px -> 32 x 32 tokens
print(f"token grid at {IMG_SIZE}px: {IMG_SIZE//meta['patch']} x {IMG_SIZE//meta['patch']}")

## 5. Features and per-patch labels

Each tile goes through the frozen backbone once. Polygons are rasterised,
downsampled onto the token grid, and each patch is labelled by how much of it
the annotation covers.

Patches straddling a polygon edge are **dropped**, not forced into a class.
Cardboard markers are punched out of the masks, because a head that learned
"white cardboard" would score well here and fail on unmarked imagery.

In [ ]:
import numpy as np
from PIL import Image

POS_T, NEG_T = 0.50, 0.10

polys = tp.read_cvat_zips([str(DATA / 'batch_*.zip')])
tiles = tp.index_tiles(DATA)
crib  = tp.read_marker_boxes(CRIB) if CRIB.exists() else {}
names = [n for n in sorted(polys) if n in tiles]
print(f"annotated tiles with imagery: {len(names)}")

recs = []
for i, n in enumerate(names, 1):
    f, (gh, gw) = vf.patch_features(model, str(tiles[n]), meta, img_size=IMG_SIZE)
    f = np.asarray(f, dtype=np.float32)
    f /= (np.linalg.norm(f, axis=1, keepdims=True) + 1e-8)
    with Image.open(tiles[n]) as im: size = im.size
    side = gh * meta['patch']
    cov  = tp.patch_coverage(tp.tile_mask(size, polys[n], crib.get(n)), side, meta['patch'])
    inst = []
    for pts in polys[n]:
        g = tp.patch_coverage(tp.tile_mask(size, [pts], crib.get(n)), side, meta['patch']) >= 0.5
        if g.any(): inst.append(g)
    recs.append(dict(name=n, feat=f, cov=cov, grid=(gh, gw), inst=inst, src=tp.source_of(n)))
    if i % 25 == 0: print(f"  {i}/{len(names)}")

print(f"done. grid {recs[0]['grid']}, dim {recs[0]['feat'].shape[1]}, frames {len({r['src'] for r in recs})}")

## 6. Split by source frame

**This matters more than anything else in the notebook.** Tiles are cut 8x6
from one 8192x5460 frame, and the bottom tile row is shifted flush to the
bottom edge, so it overlaps the row above by 684 px. Neighbouring tiles from
one frame share content, and in that band share literal pixels.

Splitting randomly over tiles or patches puts near-duplicates on both sides and
inflates every metric below. Group by source frame.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression

G = np.array([r['src'] for r in recs])
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
              .split(np.zeros(len(recs)), groups=G))

def pack(idx):
    X, Y = [], []
    for j in idx:
        r = recs[j]; y = tp.label_patches(r['cov'], POS_T, NEG_T); k = y >= 0
        X.append(r['feat'][k]); Y.append(y[k])
    return np.concatenate(X), np.concatenate(Y)

Xtr, Ytr = pack(tr)
clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0).fit(Xtr, Ytr)
print(f"train {len(set(G[tr]))} frames / {len(tr)} tiles / {len(Ytr)} patches ({100*Ytr.mean():.1f}% positive)")
print(f"test  {len(set(G[te]))} frames / {len(te)} tiles")

## 7. Metrics

- **patch** AP (= mAP, one class), ROC AUC, precision, recall, F1
- **mask** mean IoU and Dice per tile
- **instance** COCO-style mAP@[.50:.95], AP@0.50, AP@0.75 over connected components

All scored on the token grid, which is the probe's real resolution. Upsampling
predictions to pixels before scoring would flatter the boundary.

In [ ]:
ys, ps, tl = [], [], []
for j in te:
    r = recs[j]; gh, gw = r['grid']
    prob = clf.predict_proba(r['feat'])[:, 1]
    y = tp.label_patches(r['cov'], POS_T, NEG_T); k = y >= 0
    ys.append(y[k]); ps.append(prob[k])
    tl.append({'prob': prob.reshape(gh, gw), 'gt': r['cov'] >= 0.5, 'gt_instances': r['inst']})

Y = np.concatenate(ys); P = np.concatenate(ps)
res = pe.report(Y, P, tl)

### Precision / recall curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

pr, rc, _ = precision_recall_curve(Y, P)
plt.figure(figsize=(5,4))
plt.plot(rc, pr, lw=2)
plt.axhline(Y.mean(), ls='--', c='gray', lw=1, label=f'chance {Y.mean():.2f}')
plt.xlabel('recall'); plt.ylabel('precision')
plt.title(f"patch PR, AP={res['patch']['ap']:.3f}")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

### Instance AP against post-processing
A per-patch head has no spatial prior and fragments single canopies, so closing the mask before labelling components matters a lot. Check it rather than trusting the default.

In [ ]:
print(f"{'smooth':>7}{'min_size':>10}{'mAP@.5:.95':>12}{'AP@.50':>9}{'preds':>8}")
for sm in (0,1,2,3):
    for ms in (4,8,16):
        im = pe.instance_map(tl, res['patch']['threshold'], min_size=ms, smooth=sm)
        print(f"{sm:>7}{ms:>10}{im['mAP_50_95']:>12.3f}{im['AP_50']:>9.3f}{im['n_pred_instances']:>8}")
print(f"\nground truth instances: {sum(len(t['gt_instances']) for t in tl)}")

### Qualitative check
Numbers hide failure modes. Look at the heatmaps.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

show = te[:4]
fig, ax = plt.subplots(2, len(show), figsize=(4*len(show), 8))
for c, j in enumerate(show):
    r = recs[j]; gh, gw = r['grid']
    prob = clf.predict_proba(r['feat'])[:, 1].reshape(gh, gw)
    ax[0, c].imshow(Image.open(tiles[r['name']]).resize((320, 320)))
    ax[0, c].set_title(r['name'][-18:], fontsize=8); ax[0, c].axis('off')
    ax[1, c].imshow(prob, cmap='magma', vmin=0, vmax=1)
    ax[1, c].contour(r['cov'] >= 0.5, levels=[.5], colors='lime', linewidths=1.2)
    ax[1, c].set_title('p(weed), green = ground truth', fontsize=8); ax[1, c].axis('off')
plt.tight_layout(); plt.show()

## 8. Save

In [ ]:
import joblib, json
OUT = Path('/content/drive/MyDrive/chinee-apple/probe_out'); OUT.mkdir(parents=True, exist_ok=True)
joblib.dump({'clf': clf, 'family': FAMILY, 'img_size': IMG_SIZE,
             'patch': meta['patch'], 'l2_normalized': True}, OUT / 'probe.joblib')
clean = {k: {kk: (float(vv) if isinstance(vv, (int, float, np.floating)) else str(vv))
             for kk, vv in v.items() if kk != 'per_iou'} for k, v in res.items()}
(OUT / 'metrics.json').write_text(json.dumps(clean, indent=2))
print('saved to', OUT)

## 9. Predict on new imagery

`predict.py` is the inference side. Give it the saved probe and any images and
it writes a probability heatmap, a binary mask, an overlay and one row per
plant found.

Metrics need ground truth, so `--annotations` is optional. Without it you get
pictures and a plant list but no AP, because there is nothing to compare
against. An unannotated image can never produce a metric.

In [ ]:
SCRIPTS = 'chinee-apple-cv-benchmark/pipelines/dinov3-linear-probe'
PROBE   = str(OUT / 'probe.joblib')

# the CLI tools need the gated backbone passed through too
BB = f"--repo {REPO_DIR} --weights {WEIGHTS}" if FAMILY == 'dinov3' else ''

# predictions only. Works on any image, annotated or not.
cmd = (f"python {SCRIPTS}/predict.py --probe {PROBE} {BB} "
       f"--image '{DATA}/batch_02/*.jpg' --out /content/preds")
!{cmd}

# the same run, but scored, because these tiles do have polygons
cmd = (f"python {SCRIPTS}/predict.py --probe {PROBE} {BB} "
       f"--image '{DATA}/batch_02/*.jpg' --annotations '{DATA}/batch_*.zip' "
       f"--out /content/preds_scored")
!{cmd}

In [ ]:
# look at a few overlays
from PIL import Image
import glob, matplotlib.pyplot as plt
ov = sorted(glob.glob('/content/preds/*_overlay.png'))[:4]
if ov:
    fig, ax = plt.subplots(1, len(ov), figsize=(4*len(ov), 4))
    ax = np.atleast_1d(ax)
    for a, f in zip(ax, ov):
        a.imshow(Image.open(f)); a.axis('off'); a.set_title(f.split('/')[-1][-24:], fontsize=7)
    plt.tight_layout(); plt.show()

import pandas as pd
print(pd.read_csv('/content/preds/instances.csv').head(10))

cmd = (f"python {SCRIPTS}/compare.py --zips '{DATA}/batch_*.zip' "
       f"--tiles '{DATA}' --crib '{DATA}/tile_markers.csv' --family {FAMILY} {BB} "
       f"--img-sizes 448 768 --smooth 1 2 --out /content/compare_out")
!{cmd}

In [ ]:
cmd = (f"python {SCRIPTS}/compare.py --zips '{DATA}/batch_*.zip' "
       f"--tiles '{DATA}' --crib '{DATA}/tile_markers.csv' --family {FAMILY} {BB} "
       f"--img-sizes 448 768 --smooth 1 2 --out /content/compare_out")
!{cmd}

## What to expect, and where the ceiling is

Measured on this dataset with **DINOv2 ViT-B/14** (the ungated fallback, not
DINOv3), 180 annotated tiles, 56 frames, 17 frames held out:

| level | metric | value |
|---|---|---|
| patch | AP | 0.77 |
| patch | precision / recall | 0.65 / 0.81 |
| mask | IoU / Dice | 0.50 / 0.63 |
| instance | mAP@[.50:.95] | 0.15 |
| instance | AP@0.50 | 0.38 |

**Swapping the head barely matters.** Across 20 configurations, five heads by
two resolutions by two smoothing settings, patch AP spans only 0.758 to 0.771.
Logistic regression, a random forest and a small MLP land within noise of each
other. That is the expected behaviour of a strong self-supervised backbone, and
it means the simplest head is the right one.

**Resolution helps a little, not a lot.** Going 448 to 768 leaves patch AP flat
and lifts instance mAP from about 0.147 to 0.166, for roughly 3x the compute.

**Instance mAP is far below patch AP**, and that is the honest picture. This is
a decent per-patch classifier and a weak instance detector, because the head
has no spatial prior and canopies fragment. A light convolutional decoder over
the token grid is the principled fix, not threshold tuning.

**The likely real ceiling is the labels.** Only the blue-board plant was
outlined in each tile, while visually similar unlabelled scrub sits beside it,
so the model is penalised for finding plants that genuinely are there. No
amount of head swapping addresses that.